# Video I/O — `exordium.video.core.io`

| Function / Class | Purpose | Returns |
|---|---|---|
| `get_video_metadata` | Read fps, resolution, frame count | `dict` |
| `load_video` | Load all frames (optionally resampled) | `(T, 3, H, W)` uint8 tensor |
| `load_frames` | Load specific frames by index | `(N, 3, H, W)` uint8 tensor |
| `Video` | Context-manager + `iter_batches` | `(B, 3, H, W)` uint8 per batch |
| `batch_iterator` | Chunk any iterable into batches | `list` per batch |

Demo uses `multispeaker_short.mp4` (1 s, 30 fps, 1280×720).

In [1]:
from pathlib import Path

video_path = Path("../tests/fixtures/video/multispeaker_short.mp4")
print(f"Video: {video_path}")
print(f"Exists: {video_path.exists()}")

Video: ../tests/fixtures/video/multispeaker_short.mp4
Exists: True


## 1. Video Metadata

In [2]:
from exordium.video.core.io import get_video_metadata

meta = get_video_metadata(video_path)
print(f"num_frames : {meta['num_frames']}")
print(f"fps        : {meta['fps']:.2f}")
print(f"resolution : {meta['width']}x{meta['height']}")
print(f"duration   : {meta['duration']:.2f} s")

/Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


num_frames : 30
fps        : 29.97
resolution : 1280x720
duration   : 1.00 s


objc[37989]: Class AVFFrameReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x138c343a8) and /opt/homebrew/Cellar/ffmpeg/8.0.1/lib/libavdevice.62.1.100.dylib (0x334c5c328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[37989]: Class AVFAudioReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x138c343f8) and /opt/homebrew/Cellar/ffmpeg/8.0.1/lib/libavdevice.62.1.100.dylib (0x334c5c378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


## 2. Load All Frames

In [3]:
from exordium.video.core.io import load_video

# load_video returns (Tensor (T, C, H, W), actual_fps)
frames, fps = load_video(video_path)

print(f"frames: shape={tuple(frames.shape)},  dtype={frames.dtype}")
print(f"fps   : {fps:.2f}")

frames: shape=(30, 3, 720, 1280),  dtype=torch.uint8
fps   : 29.97


## 3. Load Frames from the Middle

In [4]:
n = meta["num_frames"]
start = n // 3
end = (n * 2) // 3

mid_frames, _ = load_video(video_path, start_frame=start, end_frame=end)

print(f"Middle clip: frames {start}–{end}  →  shape={tuple(mid_frames.shape)}")

Middle clip: frames 10–20  →  shape=(10, 3, 720, 1280)


## 4. Load Specific Frames by Index

In [5]:
from exordium.video.core.io import load_frames

# sample every 5th frame
frame_ids = list(range(0, n, max(1, n // 10)))  # ~10 key frames
key_frames = load_frames(video_path, frame_ids=frame_ids)

print(f"Key frames: indices={frame_ids}")
print(f"           shape={tuple(key_frames.shape)}")

Key frames: indices=[0, 3, 6, 9, 12, 15, 18, 21, 24, 27]
           shape=(10, 3, 720, 1280)


## 5. Video Context Manager + Batch Iteration

In [6]:
from exordium.video.core.io import Video

batch_size = 8

with Video(video_path) as v:
    print(f"Opened: {v.num_frames} frames, {v.fps:.1f} fps, {v.width}x{v.height}")

    for i, batch in enumerate(v.iter_batches(batch_size=batch_size)):
        # batch: Tensor (T, C, H, W)
        print(f"  Batch {i:2d}: shape={tuple(batch.shape)}")

print("Video closed.")

Opened: 30 frames, 30.0 fps, 1280x720
  Batch  0: shape=(8, 3, 720, 1280)
  Batch  1: shape=(8, 3, 720, 1280)
  Batch  2: shape=(8, 3, 720, 1280)
  Batch  3: shape=(6, 3, 720, 1280)
Video closed.


## 6. Batch-Iterate a Generic Iterable

In [7]:
from exordium.video.core.io import batch_iterator

# batch_iterator works on any iterable — useful for detections, paths, etc.
items = list(range(n))
batches = list(batch_iterator(items, batch_size=8))

print(f"Total items : {len(items)}")
print("Batch size  : 8")
print(f"Num batches : {len(batches)}")
print(f"Last batch  : {batches[-1]}  (may be smaller)")

Total items : 30
Batch size  : 8
Num batches : 4
Last batch  : [24, 25, 26, 27, 28, 29]  (may be smaller)


## 7. Resample to Lower FPS

In [8]:
# load_video accepts fps= to temporally subsample
frames_5fps, _ = load_video(video_path, fps=5)

print(f"Original: {n} frames @ {meta['fps']:.1f} fps")
print(f"At 5 fps: shape={tuple(frames_5fps.shape)}")

Original: 30 frames @ 30.0 fps
At 5 fps: shape=(5, 3, 720, 1280)
